In [17]:
# ============================================
# STEP 1: IMPORTS & SCRIPT OVERVIEW
# Description: Load required libraries and provide a high-level overview of the workflow.
# ============================================

"""
# EPC Deviation Statistical Analysis Script

This script analyses deviations between modelled and EPC-predicted energy use intensity (EUI)
for ~54 residential buildings.

## Key Steps:
1. Load and clean EPC dataset
2. Create grouped categorical variables:
   - EPC Rating (A–B, C–D, E–G)
   - Construction Age Bands (4 grouped periods)
   - Fuel Type (Electric vs Other Fuel)
   - Heating System (Heat Pump vs Other)
   - Built Form (as-is)
3. Generate visualisations:
   - Boxplots + strip plots for deviation across categories
4. Perform statistical testing:
   - Mann–Whitney U (2-group comparisons)
   - Kruskal–Wallis (multi-group comparisons)
   - Levene’s test (variance comparison)
5. Compute effect sizes:
   - Rank-biserial correlation (Mann–Whitney)
   - Eta-squared (Kruskal–Wallis)
6. Export:
   - Figures to RESULTS_FIGURES folder
   - Statistical summary CSV for reporting

All outputs are structured for use in academic/research reporting.
"""

import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

In [18]:
# ============================================
# STEP 2: LOAD DATA
# Description: Read CSV and inspect basic structure.
# ============================================

file_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_D2_and_SD2_RESULTS.csv"

df = pd.read_csv(file_path)

# Standardize column names
df.columns = df.columns.str.strip()

# Ensure key column exists
target_col = "CONSUMPTION_COMPARISON_PERCENT"

# Drop rows where target is missing
df = df.dropna(subset=[target_col])

print(f"Dataset loaded. Rows after cleaning: {len(df)}")
df.head()

Dataset loaded. Rows after cleaning: 54


,BUILDING_REFERENCE_NUMBER,OSG_REFERENCE_NUMBER,ADDRESS1,ADDRESS2,ADDRESS3,POSTCODE,LATITUDE,LONGITUDE,ORIENTATION,ORIENTATION_ESPR_ROT,...,WINDOWS_IRR_FLEX,ROOF_IRR_FLEX,WALLS_IRR_FLEX,FAB_IRR_FLEX,HP_IRR_FLEX,SOLAR_THERMAL_IRR_FLEX,PV_IRR_FLEX,BATTERY_IRR_FLEX,MODEL_CONSUMPTION_CURRENT,CONSUMPTION_COMPARISON_PERCENT
0,1001829067,122009933,19 CULCREUCH AVENUE,FINTRY,GLASGOW,G63 0YB,56.0557,-4.2233,100,170,...,-0.023127,-0.037221,0.036246,0.019304,0.135966,0.111072,NaN,NaN,575.579222,166.471862
1,1001951858,122010025,GLENVIEW,FINTRY,GLASGOW,G63 0XJ,56.0528,-4.2206,180,90,...,-0.015884,-0.074175,-0.017873,-0.009717,0.418858,0.135407,0.081951,-0.037591,310.176587,25.071204
2,1234761001,122009968,1 MENZIES TERRACE,FINTRY,GLASGOW,G63 0YJ,56.0584,-4.2248,150,120,...,-0.079761,-0.058603,-0.003530,0.003958,-0.012701,0.005398,0.073403,-0.029204,527.270124,79.955674
3,1001664929,122038906,18 DUNMORE GARDENS,FINTRY,GLASGOW,G63 0XN,56.0520,-4.2225,135,135,...,-0.013642,-0.080009,-0.012871,-0.007177,0.225289,0.134885,0.085307,-0.025672,309.432959,-17.484544
4,1001829059,122009928,11 CULCREUCH AVENUE,FINTRY,GLASGOW,G63 0YB,56.0561,-4.2247,210,60,...,-0.069174,-0.056404,-0.002785,-0.028142,-0.049844,0.019691,0.119490,-0.033834,425.486187,57.005973


In [19]:
# ============================================
# STEP 3: CREATE GROUPED VARIABLES
# Description: Generate categorical groupings as specified.
# ============================================

# --- EPC Rating Grouping ---
def group_epc(rating):
    if pd.isna(rating):
        return np.nan
    rating = str(rating).upper()
    if rating in ["A", "B"]:
        return "A_to_B"
    elif rating in ["C", "D"]:
        return "C_to_D"
    elif rating in ["E", "F", "G"]:
        return "E_to_G"
    return np.nan

df["EPC_GROUP"] = df["CURRENT_ENERGY_RATING"].apply(group_epc)

# --- Construction Age Grouping ---
def group_age(age):
    if pd.isna(age):
        return np.nan
    age = str(age).lower()

    if "before 1919" in age or "1930-1949" in age:
        return "Before 1919 to 1949"
    elif "1950-1964" in age or "1965-1975" in age:
        return "1950-1975"
    elif "1976-1983" in age or "1984-1991" in age:
        return "1976-1991"
    elif any(x in age for x in ["1992-1998", "1999-2002", "2003-2007", "2008"]):
        return "1992 onwards"
    return np.nan

df["AGE_GROUP"] = df["CONSTRUCTION_AGE_BAND"].apply(group_age)

# --- Fuel Grouping ---
def group_fuel(fuel):
    if pd.isna(fuel):
        return np.nan
    fuel = str(fuel).lower()
    if fuel in ["electricity", "electricity: electricity, unspecified tariff"]:
        return "Electric"
    return "Other Fuel"

df["FUEL_GROUP"] = df["MAIN_FUEL"].apply(group_fuel)

# --- Heating System Grouping ---
def group_heating(system):
    if pd.isna(system):
        return np.nan
    if "heat pump" in str(system).lower():
        return "Heat Pump"
    return "Other Heating System"

df["HEATING_GROUP"] = df["MAIN_HEATING_CATEGORY"].apply(group_heating)

# --- Built Form (no change, just rename for consistency) ---
df["BUILT_FORM_GROUP"] = df["BUILT_FORM"]

print("Grouping complete.")

Grouping complete.


In [ ]:
# ============================================
# STEP 4: CREATE OUTPUT DIRECTORY
# Description: Ensure figure output folder exists.
# ============================================

fig_dir = "/Users/jakegehrung/Desktop/data_raw/RESULTS_FIGURES/BASELINE_RESULTS"
os.makedirs(fig_dir, exist_ok=True)

print(f"Figures will be saved to: {fig_dir}")

Figures will be saved to: /Users/jakegehrung/Desktop/data_raw/RESULTS_FIGURES


In [21]:
# ============================================
# STEP 5: PLOTTING FUNCTION
# Description: Generate boxplot + stripplot for each grouping.
# ============================================

def plot_group(df, group_col, target_col):
    plt.figure(figsize=(8, 5))
    
    sns.boxplot(data=df, x=group_col, y=target_col)
    sns.stripplot(data=df, x=group_col, y=target_col, alpha=0.6)
    
    plt.title(f"{target_col} by {group_col}")
    plt.xticks(rotation=30)
    plt.tight_layout()
    
    file_name = f"{group_col}_boxplot.png"
    plt.savefig(os.path.join(fig_dir, file_name), dpi=300)
    plt.close()

# Generate plots
group_cols = ["EPC_GROUP", "AGE_GROUP", "FUEL_GROUP", "HEATING_GROUP", "BUILT_FORM_GROUP"]

for col in group_cols:
    plot_group(df, col, target_col)

print("All plots saved.")

All plots saved.


In [22]:
# ============================================
# STEP 6: STATISTICAL TEST FUNCTIONS
# Description: Define Mann–Whitney, Kruskal–Wallis, Levene, and effect sizes.
# ============================================

def mannwhitney_with_effect(df, group_col, target):
    groups = df[group_col].dropna().unique()
    g1, g2 = groups
    
    x = df[df[group_col] == g1][target].dropna()
    y = df[df[group_col] == g2][target].dropna()
    
    stat, p = stats.mannwhitneyu(x, y, alternative='two-sided')
    
    # Rank-biserial correlation
    n1, n2 = len(x), len(y)
    rbc = 1 - (2 * stat) / (n1 * n2)
    
    return stat, p, rbc, x.median(), y.median()

def kruskal_with_effect(df, group_col, target):
    groups = [g[target].dropna() for _, g in df.groupby(group_col)]
    
    stat, p = stats.kruskal(*groups)
    
    n = sum(len(g) for g in groups)
    k = len(groups)
    
    eta_sq = (stat - k + 1) / (n - k)
    
    medians = df.groupby(group_col)[target].median().to_dict()
    
    return stat, p, eta_sq, medians

def levene_test(df, group_col, target):
    groups = [g[target].dropna() for _, g in df.groupby(group_col)]
    stat, p = stats.levene(*groups)
    return stat, p

In [23]:
# ============================================
# STEP 7: RUN STATISTICAL ANALYSIS
# Description: Execute tests and store results.
# ============================================

results = []

# Two-group tests
two_group_cols = ["FUEL_GROUP", "HEATING_GROUP", "BUILT_FORM_GROUP"]

for col in two_group_cols:
    try:
        stat, p, effect, med1, med2 = mannwhitney_with_effect(df, col, target_col)
        lev_stat, lev_p = levene_test(df, col, target_col)
        
        results.append({
            "Grouping": col,
            "Test": "Mann-Whitney U",
            "Statistic": stat,
            "p-value": p,
            "Effect Size": effect,
            "Levene Stat": lev_stat,
            "Levene p": lev_p,
            "Group Medians": f"{med1:.2f}, {med2:.2f}"
        })
    except Exception as e:
        print(f"Error in {col}: {e}")

# Multi-group tests
multi_group_cols = ["EPC_GROUP", "AGE_GROUP"]

for col in multi_group_cols:
    try:
        stat, p, effect, medians = kruskal_with_effect(df, col, target_col)
        lev_stat, lev_p = levene_test(df, col, target_col)

        rounded_medians = {k: round(v, 2) for k, v in medians.items()}

        results.append({
            "Grouping": col,
            "Test": "Kruskal-Wallis",
            "Statistic": stat,
            "p-value": p,
            "Effect Size": effect,
            "Levene Stat": lev_stat,
            "Levene p": lev_p,
            "Group Medians": str(rounded_medians)
        })
    except Exception as e:
        print(f"Error in {col}: {e}")

results_df = pd.DataFrame(results)

results_df

,Grouping,Test,Statistic,p-value,Effect Size,Levene Stat,Levene p,Group Medians
0,FUEL_GROUP,Mann-Whitney U,51.000000,5.726778e-07,0.842593,0.166354,0.685047,"-26.73, 75.58"
1,HEATING_GROUP,Mann-Whitney U,576.000000,4.838523e-05,-0.662338,11.481669,0.001347,"31.13, -39.30"
2,BUILT_FORM_GROUP,Mann-Whitney U,490.000000,1.549429e-02,-0.392045,0.045275,0.832332,"-6.56, -31.70"
3,EPC_GROUP,Kruskal-Wallis,2.062987,3.564742e-01,0.001235,1.252930,0.294310,"{'A_to_B': -17.5, 'C_to_D': -22.42, 'E_to_G': ..."
4,AGE_GROUP,Kruskal-Wallis,9.103550,2.794535e-02,0.122071,7.304280,0.000373,"{'1950-1975': 45.05, '1976-1991': -33.52, '199..."


In [ ]:
# ============================================
# STEP 8: SAVE RESULTS
# Description: Export statistical summary to CSV.
# ============================================

output_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_BASELINE_STATS.csv"

results_df.to_csv(output_path, index=False)

print(f"Statistical results saved to: {output_path}")

Statistical results saved to: /Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_BASELINE_STATS.csv
